In [1]:
%cd ..
%load_ext autoreload
%autoreload 2

/home/dongmin/userdata/open-score-string-quartets


/home/dongmin/.local/share/virtualenvs/open-score-string-quartets-wd2Cnojv/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
import os
import sys
import subprocess
import warnings
from typing import Union, Any, Optional
import shutil
from pathlib import Path
import re
import difflib
from collections import Counter, defaultdict
from operator import itemgetter, contains, eq # contains(A, B) == (B in A), eq(A, B) == (A == B)
from itertools import groupby
from tempfile import NamedTemporaryFile, TemporaryDirectory

import math
import random

import json
import csv
import strictyaml as syaml
import pandas as pd

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

import cv2
import numpy as np
import music21 as m21
import converter21
import kernpy as kp

import xml.etree.ElementTree as ET
from modules.lmxe.lmxe import delinearize_lmxe
from modules.lmxe.lmxe.utils import split_tokens_by_key_token, find_nth_token_index, single_line_to_multi_line_lmxe
from modules.lmxe.lmxe.LMXEFile import LMXEFile, LMXEMetadata
from modules.lmxe.lmxe.vocabulary import (
  KEY_TOKENS,
  BEATS_TOKENS, 
  BEAT_TYPE_TOKENS, 
  TIME_SIGNATURE_TOKENS, 
  CLEF_TOKENS, 
  NOTE_TYPE_TOKENS, 
  PITCH_TOKENS, 
  VOICE_TOKENS, 
  TIME_MODIFICATION_TOKENS, 
  ACCIDENTAL_TOKENS, 
  STEM_TOKENS, 
  STAFF_TOKENS, 
  BEAM_TOKENS, 
  TREMOLO_MARKS_TOKENS, 
  TREMOLO_TYPE_TOKENS, 
  TREMOLO_TOKENS, 
  EXTENDED_FLAVOR_TOKENS, 
)
from modules.lmxe.lmxe.evaluation.omr_ned import calc_omr_ned
from scripts.utils import load_ossq_metadata

from IPython.core.magic import register_cell_magic
@register_cell_magic
def skip(line, cell):
  return

In [3]:
def merge_partwise_lmxe(pwlmxe:list[LMXEFile|Path]) -> LMXEFile:
  """Merge part-wise LMXE into multi-part LMXEFile"""
  # load LMXE files if Path is given
  pwlmxe = [
    LMXEFile.load(pl) if isinstance(pl, (str, Path)) else pl
    for pl in pwlmxe
  ]
  
  lmxe = LMXEFile()
  
  max_n_measures = max([ 
    len(pl.measures)
    for pl in pwlmxe 
  ])
  
  for pl in pwlmxe:
    # pad measures with empty measures
    n_measures = len(pl.measures)
    if n_measures < max_n_measures:
      pl.measures += [ 'measure' for _ in range(max_n_measures - n_measures) ]
  
  # iterate mesure-wise
  for parts in zip(*[pl.measures for pl in pwlmxe]):
    measure = ['measure']
    
    for p_i, m in enumerate(parts):
      m = m.split()[1:] # omit 'measure' token
      m = [f'part:{p_i + 1}', 'multi'] + m
      measure += m
    
    lmxe.measures.append(' '.join(measure))
  
  lmxe.metadata = LMXEMetadata(
    number_of_measures=len(lmxe.measures) if lmxe.measures else 1,
    times=[[-1,-1]]
  )
  
  return lmxe

## Load metadata

In [4]:
# excluded scores and segments from ossq v2.2.1
# total: 379 segments
excluded_scores = { 'sq7358708', 'sq7267316', 'sq7471661' }
excluded_segments = {
  # kern errors: 18
  'sq7383977:0013:0004',
  'sq7383977:0014:0002',
  'sq7383977:0014:0003',
  'sq7383977:0016:0001',
  'sq7383977:0016:0003',
  'sq7383977:0016:0004',
  'sq7295726:0026:0003',
  'sq8741475:0006:0002',
  'sq8454356:0001:0004',
  'sq8454356:0005:0005',
  'sq8454356:0007:0003',
  'sq8454356:0016:0002',
  'sq8811375:0051:0002',
  'sq8811375:0017:0002',
  'sq8928855:0006:0005',
  'sq8823783:0030:0001',
  'sq8885439:0013:0003',
  'sq9631717:0021:0004',
  # staff split errors: 1
  'sq10675759:0028:0002',
  # double dotted notes: 281
  # cause error in kern-ekern conversion
  'sq7313978:0014:0004',
  'sq7313978:0005:0004',
  'sq7313978:0012:0002',
  'sq7313978:0026:0003',
  'sq7313978:0015:0002',
  'sq13744399:0012:0002',
  'sq13744399:0013:0003',
  'sq13744399:0012:0001',
  'sq13744399:0013:0004',
  'sq7295726:0003:0001',
  'sq7295726:0004:0005',
  'sq7295726:0006:0005',
  'sq7295726:0027:0003',
  'sq7295726:0011:0003',
  'sq7295726:0004:0002',
  'sq7295726:0011:0002',
  'sq7295726:0007:0001',
  'sq7295726:0011:0004',
  'sq7295726:0002:0001',
  'sq7295726:0012:0001',
  'sq8741475:0040:0003',
  'sq8741475:0041:0002',
  'sq8741475:0040:0002',
  'sq8741475:0043:0004',
  'sq8741475:0036:0001',
  'sq8741475:0035:0005',
  'sq8741475:0039:0001',
  'sq8741475:0041:0003',
  'sq14387632:0001:0004',
  'sq14387632:0017:0004',
  'sq10527526:0012:0001',
  'sq8853405:0009:0005',
  'sq9396439:0012:0003',
  'sq9396439:0012:0001',
  'sq9719026:0014:0001',
  'sq9719026:0018:0003',
  'sq9719026:0014:0002',
  'sq7872392:0015:0002',
  'sq7872392:0015:0001',
  'sq7108150:0003:0002',
  'sq7302602:0004:0003',
  'sq8928855:0011:0001',
  'sq8928855:0011:0003',
  'sq8938822:0027:0001',
  'sq9094235:0024:0002',
  'sq9094235:0017:0002',
  'sq9094235:0005:0001',
  'sq9094235:0022:0004',
  'sq9094235:0023:0001',
  'sq9094235:0007:0003',
  'sq9094235:0024:0004',
  'sq9094235:0011:0001',
  'sq9094235:0016:0003',
  'sq9094235:0021:0004',
  'sq9094235:0024:0003',
  'sq9094235:0017:0003',
  'sq11164006:0001:0001',
  'sq11164006:0021:0002',
  'sq11164006:0014:0003',
  'sq11164006:0017:0005',
  'sq11164006:0020:0004',
  'sq11164006:0016:0004',
  'sq11164006:0022:0001',
  'sq11164006:0020:0001',
  'sq11164006:0016:0003',
  'sq11164006:0018:0002',
  'sq11164006:0015:0003',
  'sq11164006:0016:0005',
  'sq11164006:0022:0002',
  'sq11164006:0020:0005',
  'sq11164006:0018:0003',
  'sq11164006:0013:0003',
  'sq11164006:0016:0002',
  'sq11164006:0017:0002',
  'sq11164006:0013:0001',
  'sq11164006:0018:0001',
  'sq11164006:0013:0002',
  'sq11164006:0021:0001',
  'sq11164006:0015:0001',
  'sq11164006:0021:0005',
  'sq11164006:0013:0004',
  'sq11164006:0020:0003',
  'sq11164006:0021:0003',
  'sq11164006:0017:0004',
  'sq11164006:0020:0002',
  'sq11164006:0015:0005',
  'sq11164006:0015:0002',
  'sq11164006:0017:0001',
  'sq11164006:0021:0004',
  'sq11164006:0022:0004',
  'sq11164006:0014:0002',
  'sq11164006:0017:0003',
  'sq11164006:0014:0001',
  'sq8885439:0004:0004',
  'sq14720995:0011:0003',
  'sq14720995:0009:0001',
  'sq14720995:0012:0002',
  'sq14720995:0005:0004',
  'sq14720995:0013:0003',
  'sq14720995:0004:0002',
  'sq14720995:0001:0003',
  'sq14720995:0009:0004',
  'sq14720995:0011:0001',
  'sq14720995:0009:0003',
  'sq7127785:0001:0001',
  'sq7127785:0001:0002',
  'sq7127785:0006:0004',
  'sq7127785:0001:0003',
  'sq7127785:0007:0001',
  'sq7127785:0006:0003',
  'sq10372717:0005:0004',
  'sq10372717:0001:0001',
  'sq10307350:0013:0003',
  'sq10307350:0015:0002',
  'sq10307350:0011:0003',
  'sq10307350:0014:0005',
  'sq10307350:0015:0001',
  'sq10307350:0014:0001',
  'sq10307350:0011:0002',
  'sq10307350:0013:0005',
  'sq10307350:0015:0003',
  'sq10307350:0013:0002',
  'sq10307350:0014:0002',
  'sq10307350:0011:0001',
  'sq10307350:0013:0001',
  'sq10307350:0015:0004',
  'sq10307350:0014:0003',
  'sq10307350:0012:0002',
  'sq10307350:0012:0004',
  'sq10307350:0012:0005',
  'sq10307350:0013:0004',
  'sq10307350:0014:0004',
  'sq10307350:0012:0003',
  'sq10307350:0012:0001',
  'sq10307350:0011:0004',
  'sq7249986:0013:0003',
  'sq15049456:0017:0003',
  'sq15049456:0022:0002',
  'sq10313029:0011:0003',
  'sq10313029:0010:0004',
  'sq10313029:0011:0004',
  'sq10313029:0017:0001',
  'sq7158117:0002:0002',
  'sq7158117:0022:0001',
  'sq7158117:0006:0001',
  'sq7158117:0005:0004',
  'sq7384409:0001:0002',
  'sq7524617:0003:0001',
  'sq7524617:0006:0002',
  'sq7588853:0012:0001',
  'sq7588853:0014:0002',
  'sq7588853:0014:0004',
  'sq7556360:0014:0005',
  'sq7556360:0016:0001',
  'sq7556360:0015:0001',
  'sq7556360:0016:0002',
  'sq7082029:0001:0003',
  'sq7082029:0003:0001',
  'sq7082029:0001:0001',
  'sq7082029:0001:0003',
  'sq7082029:0003:0001',
  'sq7082029:0001:0001',
  'sq7082029:0001:0003',
  'sq7082029:0003:0001',
  'sq7082029:0001:0001',
  'sq7082029:0001:0003',
  'sq7082029:0003:0001',
  'sq7082029:0001:0001',
  'sq8885571:0008:0004',
  'sq8885571:0002:0003',
  'sq8885571:0005:0002',
  'sq8885571:0024:0005',
  'sq8623643:0004:0002',
  'sq7302710:0004:0003',
  'sq7302710:0005:0003',
  'sq7302710:0002:0001',
  'sq7302710:0004:0004',
  'sq7302710:0007:0002',
  'sq7302710:0002:0002',
  'sq7353137:0007:0003',
  'sq7353137:0008:0002',
  'sq7353137:0007:0001',
  'sq7353137:0008:0004',
  'sq16138966:0002:0004',
  'sq16138966:0002:0003',
  'sq16138966:0005:0004',
  'sq16138966:0005:0003',
  'sq15730717:0007:0001',
  'sq15730717:0008:0005',
  'sq15624112:0005:0002',
  'sq15624112:0001:0003',
  'sq8940236:0027:0001',
  'sq8940236:0016:0003',
  'sq8940236:0006:0001',
  'sq8940236:0003:0002',
  'sq8940236:0013:0003',
  'sq8940236:0023:0003',
  'sq8940236:0026:0001',
  'sq8940236:0009:0004',
  'sq8940236:0031:0001',
  'sq7397765:0002:0005',
  'sq7397765:0009:0001',
  'sq7397765:0009:0003',
  'sq7397765:0009:0005',
  'sq7397765:0003:0004',
  'sq7397765:0017:0002',
  'sq7397765:0003:0005',
  'sq7397765:0011:0005',
  'sq7397765:0007:0004',
  'sq7397765:0007:0003',
  'sq7397765:0004:0005',
  'sq7397765:0007:0005',
  'sq7397765:0007:0001',
  'sq7397765:0003:0001',
  'sq7397765:0010:0005',
  'sq7397765:0010:0001',
  'sq7397765:0006:0001',
  'sq7397765:0003:0002',
  'sq7397765:0010:0004',
  'sq7397765:0011:0004',
  'sq7397765:0007:0002',
  'sq7397765:0018:0003',
  'sq7397765:0010:0003',
  'sq7397765:0004:0004',
  'sq7397765:0009:0002',
  'sq7397765:0004:0001',
  'sq7397765:0009:0004',
  'sq7397765:0008:0004',
  'sq7397765:0008:0005',
  'sq7397765:0003:0003',
  'sq7397765:0005:0005',
  'sq7397765:0011:0002',
  'sq7397765:0005:0001',
  'sq8630159:0023:0003',
  'sq9010547:0015:0002',
  'sq9010547:0023:0004',
  'sq9010547:0014:0001',
  'sq9010547:0014:0004',
  'sq9010547:0015:0001',
  'sq9010547:0016:0002',
  'sq9010547:0013:0004',
  'sq9010547:0001:0003',
  'sq9010547:0001:0002',
  'sq7555331:0021:0001',
  'sq7555331:0018:0002',
  'sq7555331:0015:0003',
  'sq7555331:0027:0002',
  'sq7555331:0020:0004',
  'sq7555331:0029:0004',
  'sq7555331:0013:0005',
  'sq10517302:0004:0003',
  'sq10517302:0010:0005',
  'sq10517302:0006:0003',
  'sq10517302:0004:0002',
  'sq10517302:0002:0005',
  'sq10517302:0003:0002',
  'sq10517302:0003:0001',
  'sq8823783:0034:0002',
  'sq8823783:0020:0002',
  'sq8823783:0016:0002',
  'sq8823783:0015:0003',
  'sq8823783:0027:0001',
  'sq8823783:0015:0004',
  'sq8823783:0020:0004',
  'sq8823783:0020:0003',
  'sq8823783:0001:0004',
  'sq8823783:0001:0002',
  'sq8823783:0008:0003',
  'sq8823783:0020:0001',
  'sq8823783:0008:0004',
  'sq8823783:0001:0003',
  'sq8823783:0002:0001',
  'sq8823783:0020:0005',
  'sq8823783:0015:0001',
  'sq8823783:0015:0002',
  'sq8823783:0035:0004',
  'sq8823783:0028:0003',
  'sq8823783:0021:0001',
  'sq8823783:0017:0003',
  'sq8823783:0017:0001',
  'sq8823783:0001:0001',
  # non-parsable kern segments: 79
  'sq7295726:0020:0002',
  'sq7295726:0020:0003',
  'sq8741475:0022:0004',
  'sq8741475:0023:0001',
  'sq8741475:0023:0002',
  'sq8741475:0023:0004',
  'sq8741475:0024:0001',
  'sq8741475:0024:0002',
  'sq7108150:0018:0004',
  'sq7108150:0018:0005',
  'sq7108150:0019:0001',
  'sq7108150:0019:0003',
  'sq7302602:0016:0001',
  'sq7302602:0027:0002',
  'sq8928855:0002:0001',
  'sq8928855:0002:0003',
  'sq8928855:0007:0001',
  'sq8928855:0007:0003',
  'sq8938822:0035:0004',
  'sq9094235:0001:0001',
  'sq9094235:0023:0002',
  'sq11164006:0027:0003',
  'sq11164006:0028:0005',
  'sq8885439:0001:0001',
  'sq10406164:0001:0002',
  'sq10406164:0012:0002',
  'sq9631717:0001:0001',
  'sq9631717:0032:0003',
  'sq9631717:0041:0004',
  'sq9631717:0045:0003',
  'sq7123582:0003:0001',
  'sq7123582:0003:0002',
  'sq7123582:0006:0003',
  'sq8088531:0011:0005',
  'sq8509238:0004:0002',
  'sq8509238:0009:0003',
  'sq7577795:0019:0001',
  'sq7577795:0019:0002',
  'sq7158117:0001:0001',
  'sq7158117:0001:0002',
  'sq7158117:0001:0004',
  'sq7158117:0002:0003',
  'sq7158117:0004:0005',
  'sq7158117:0007:0004',
  'sq7158117:0015:0005',
  'sq7158117:0027:0005',
  'sq7158117:0028:0001',
  'sq7158117:0028:0003',
  'sq8818128:0018:0003',
  'sq8818128:0018:0004',
  'sq8818128:0019:0004',
  'sq8818128:0020:0001',
  'sq7224846:0011:0005',
  'sq8482283:0004:0002',
  'sq8482283:0004:0003',
  'sq8482283:0004:0004',
  'sq8482283:0011:0003',
  'sq8482283:0011:0005',
  'sq8482283:0019:0001',
  'sq8482283:0027:0001',
  'sq8482283:0036:0003',
  'sq8482283:0036:0004',
  'sq8482283:0037:0001',
  'sq8482283:0045:0004',
  'sq8630159:0010:0005',
  'sq8630159:0011:0001',
  'sq8630159:0011:0002',
  'sq8630159:0011:0003',
  'sq8630159:0024:0001',
  'sq8630159:0024:0002',
  'sq8630159:0024:0003',
  'sq8630159:0024:0004',
  'sq8630159:0025:0001',
  'sq8630159:0025:0002',
  'sq8630159:0025:0003',
  'sq7555331:0007:0001',
  'sq7555331:0037:0005',
  'sq7555331:0041:0002',
  'sq10517302:0035:0002'
}

In [5]:
metadata_dict = load_ossq_metadata(Path.cwd() / 'data' / 'scores.yaml')

ossq_metadata = {}

for key, body in metadata_dict.items():
  if body['sqid'] in excluded_scores:
    continue  
  ossq_metadata[key] = body

len(ossq_metadata)

119

In [6]:
sq_dataset_dir = Path.home() / 'userdata' / 'open-score-string-quartets' / 'scores'
sq_dataset_dir.exists()

True

## Load Test segments

In [7]:
def get_random_segments(*, metadata:dict, n:int, random_seed:int=251202) -> dict:
  """
  select n random segments from each score in metadata
  Args:
    metadata: dict[imslp_id, score_metadata]
    n: int, number of random segments to select from each score
    random_seed: int, random seed for reproducibility
  """
  random.seed(random_seed)
  
  random_segments = []
  for _, body in metadata.items():
    sqid = body['sqid']
    score_dir = sq_dataset_dir / body['path']
    lmxe_dir = score_dir / 'lmxe'
    
    lmxe_paths = [ l_p for l_p in lmxe_dir.glob('*.lmxe') if l_p.stem not in excluded_segments ]
    
    sampled_lmxe = random.sample(lmxe_paths, min(n, len(lmxe_paths)))
    sampled_lmxe = sorted(sampled_lmxe)
    
    image_dir = score_dir / 'images' / 'synthetic' / 'crop_resized'
    musicxml_dir = score_dir / 'musicxml'
    kern_dir = score_dir / 'krn'
    pwlmxe_dir = score_dir / 'partwise_lmxe'
    pwkern_dir = score_dir / 'partwise_ekrn'
    
    for l_p in sampled_lmxe:
      segment_id = l_p.stem
      i_p = image_dir / f"{segment_id}.png"
      m_p = musicxml_dir / f"{segment_id}.musicxml"
      k_p = kern_dir / f"{segment_id}.ekrn"
      pl_p = sorted(pwlmxe_dir.glob(f"{segment_id}*.lmxe"))
      pk_p = sorted(pwkern_dir.glob(f"{segment_id}*.ekrn"))
      
      random_segments.append({
        'sqid': sqid,
        'path': body['path'],
        'segment_id': segment_id,
        'image_path': i_p,
        'musicxml_path': m_p,
        'lmxe_path': l_p,
        'kern_path': k_p,
        'pwlmxe_paths': pl_p,
        'pwkern_paths': pk_p,
      })
    
  return random_segments

In [8]:
test_segments = get_random_segments(metadata=ossq_metadata, n=3)
len(test_segments)

357

In [9]:
random.seed(42)
debug_sample = random.choice(test_segments)

for key, value in debug_sample.items():
  if isinstance(value, list):
    print(f"{key}: ")
    for v in value:
      print(f"    {str(v)}")
  else:
    print(f"{key}: {value}")

sqid: sq7397765
path: Schubert,_Franz/String_Quartet_in_D_minor,_D.810,_Op.14_(“Death_and_the_Maiden”)
segment_id: sq7397765:0013:0002
image_path: /home/dongmin/userdata/open-score-string-quartets/scores/Schubert,_Franz/String_Quartet_in_D_minor,_D.810,_Op.14_(“Death_and_the_Maiden”)/images/synthetic/crop_resized/sq7397765:0013:0002.png
musicxml_path: /home/dongmin/userdata/open-score-string-quartets/scores/Schubert,_Franz/String_Quartet_in_D_minor,_D.810,_Op.14_(“Death_and_the_Maiden”)/musicxml/sq7397765:0013:0002.musicxml
lmxe_path: /home/dongmin/userdata/open-score-string-quartets/scores/Schubert,_Franz/String_Quartet_in_D_minor,_D.810,_Op.14_(“Death_and_the_Maiden”)/lmxe/sq7397765:0013:0002.lmxe
kern_path: /home/dongmin/userdata/open-score-string-quartets/scores/Schubert,_Franz/String_Quartet_in_D_minor,_D.810,_Op.14_(“Death_and_the_Maiden”)/krn/sq7397765:0013:0002.ekrn
pwlmxe_paths: 
    /home/dongmin/userdata/open-score-string-quartets/scores/Schubert,_Franz/String_Quartet_in_D_m

## Partwise LMXE

In [9]:
# KEY_TOKENS = set(KEY_TOKENS)
# BEATS_TOKENS = set(BEATS_TOKENS)
# BEAT_TYPE_TOKENS = set(BEAT_TYPE_TOKENS)
# TIME_SIGNATURE_TOKENS = set(TIME_SIGNATURE_TOKENS)
# CLEF_TOKENS = set(CLEF_TOKENS)
# NOTE_TYPE_TOKENS = set(NOTE_TYPE_TOKENS)
# EXTENDED_FLAVOR_TOKENS = set(EXTENDED_FLAVOR_TOKENS)

STEM_TOKENS = set(STEM_TOKENS)
ACCIDENTAL_TOKENS = set(ACCIDENTAL_TOKENS)
PITCH_TOKENS = set(PITCH_TOKENS)
try:
  PITCH_TOKENS.remove("rest")
except:
  pass

SET_TO_TYPE = {
  # str(KEY_TOKENS): 'KEY_TOKENS', 
  # str(BEATS_TOKENS): 'BEATS_TOKENS', 
  # str(BEAT_TYPE_TOKENS): 'BEAT_TYPE_TOKENS', 
  # str(TIME_SIGNATURE_TOKENS): 'TIME_SIGNATURE_TOKENS', 
  # str(CLEF_TOKENS): 'CLEF_TOKENS', 
  # str(NOTE_TYPE_TOKENS): 'NOTE_TYPE_TOKENS', 
  # str(EXTENDED_FLAVOR_TOKENS): 'EXTENDED_FLAVOR_TOKENS', 
  str(STEM_TOKENS): 'STEM_TOKENS', 
  str(ACCIDENTAL_TOKENS): 'ACCIDENTAL_TOKENS', 
  str(PITCH_TOKENS): 'PITCH_TOKENS', 
}

def get_corrupt_methods(token:str, remove=True) -> str:
  for token_set in [
    # KEY_TOKENS,
    # BEATS_TOKENS, 
    # BEAT_TYPE_TOKENS, 
    # TIME_SIGNATURE_TOKENS, 
    # CLEF_TOKENS, 
    # NOTE_TYPE_TOKENS, 
    # EXTENDED_FLAVOR_TOKENS, 
    STEM_TOKENS, 
    ACCIDENTAL_TOKENS, 
    PITCH_TOKENS, 
  ]:
    if token in token_set:
      token_type = SET_TO_TYPE[str(token_set)]
      currupt_methods = list(token_set)
      if remove:
        currupt_methods.append('')
        
      return token_type, currupt_methods
  
  currupt_methods = [token]
  if remove:
    currupt_methods.append('')
  
  return 'UNKNOWN', currupt_methods


def corrupt_measure(token_seq: list[str], n:int, remove:bool=True, random_seed: Optional[int]=251217) -> list[str]:
  """
  Corrupt n tokens in the token sequence.'
  """
  
  if random_seed is not None:
    random.seed(random_seed)
  
  measure, token_seq = token_seq[:1], token_seq[1:]
  
  currpted_tokens = [] # list of (index, token_type, original_token, corrupted_token)
  
  token_indices = list(range(len(token_seq)))
  corrupt_indices = random.sample(token_indices, min(n, len(token_indices)))
  
  for idx in corrupt_indices:
    cur = token_seq[idx]
    token_type, corrupt_methods = get_corrupt_methods(cur, remove=remove)
    currupted_token = random.choice(corrupt_methods)
    
    token_seq[idx] = currupted_token
    
    currpted_tokens.append( (idx, token_type, cur, currupted_token) )
  
  token_seq = measure + token_seq
  
  return token_seq, currpted_tokens

In [10]:
def compare_omr_ned(sample, *, n=5, remove=False, temp_dir=None):  
  if not temp_dir:
    temp_dir = Path.cwd() / 'temp'
  
  pwlmxes = []
  for pwlmxe_path in sample['pwlmxe_paths']:
    pwlmxe = LMXEFile.load(pwlmxe_path)
    pwlmxes.append(pwlmxe)

  pwlmxes_corrupted = []
  for pwlmxe in pwlmxes:
    corrupted_measures = []
    corrupted_tokens = []
    for measure in pwlmxe.measures:
      token_seq = measure.split(' ')
      corrupted_token_seq, currpted_tokens_measure = corrupt_measure(token_seq, n=n, remove=remove)
      corrupted_measure = ' '.join(corrupted_token_seq)
      corrupted_measures.append(corrupted_measure)
      corrupted_tokens.append(currpted_tokens_measure)
    
    corrupted_pwlmxe = LMXEFile()
    corrupted_pwlmxe.metadata = pwlmxe.metadata
    corrupted_pwlmxe.measures = corrupted_measures
    corrupted_pwlmxe.corrupted_tokens = corrupted_tokens
    
    pwlmxes_corrupted.append(corrupted_pwlmxe)
  
  pwmusicxmls = []
  for i, pwlmxe in enumerate(pwlmxes):
    pwmusicxml_path = temp_dir / f"{sample['pwlmxe_paths'][i].stem}.musicxml"
    LMXEFile.write(
      pwmusicxml_path.with_suffix('.lmxe'),
      '\n'.join(pwlmxe.measures),
      pwlmxe.metadata,
    )
    pwmusicxml = delinearize_lmxe(pwlmxe, score_type='single')
    pwmusicxml.write(pwmusicxml_path)
    
    pwmusicxmls.append(pwmusicxml_path)
  
  pwmusicxmls_corrupted = []
  for i, pwlmxe in enumerate(pwlmxes_corrupted):
    pwmusicxml_path = temp_dir / f"{sample['pwlmxe_paths'][i].stem}_c.musicxml"
    LMXEFile.write(
      pwmusicxml_path.with_suffix('.lmxe'),
      '\n'.join(pwlmxe.measures),
      pwlmxe.metadata,
    )
    pwmusicxml = delinearize_lmxe(pwlmxe, score_type='single')
    pwmusicxml.write(pwmusicxml_path)
    
    pwmusicxmls_corrupted.append(pwmusicxml_path)
  
  
  pw_metrics = calc_omr_ned(
    pwmusicxmls,
    max_length=0,
    prediction_paths=pwmusicxmls_corrupted,
    ground_truth_paths=pwmusicxmls,
    output_file_path=temp_dir / f"{sample['lmxe_path'].stem}_pw.tsv",
  )
  
  musicxml_path = temp_dir / f"{sample['lmxe_path'].stem}.musicxml"
  merged_lmxe = merge_partwise_lmxe(pwlmxes)
  LMXEFile.write(
    musicxml_path.with_suffix('.lmxe'),
    '\n'.join(merged_lmxe.measures),
    merged_lmxe.metadata,
  )
  musicxml = delinearize_lmxe(merged_lmxe, score_type='multi')
  musicxml.write(musicxml_path)
  
  musicxml_corrupted_path = temp_dir / f"{sample['lmxe_path'].stem}_c.musicxml"
  merged_lmxe = merge_partwise_lmxe(pwlmxes_corrupted)
  LMXEFile.write(
    musicxml_corrupted_path.with_suffix('.lmxe'),
    '\n'.join(merged_lmxe.measures),
    merged_lmxe.metadata,
  )
  musicxml = delinearize_lmxe(merged_lmxe, score_type='multi')
  musicxml.write(musicxml_corrupted_path)
  
  metrics = calc_omr_ned(
    [musicxml_path],
    max_length=0,
    prediction_paths=[musicxml_corrupted_path],
    ground_truth_paths=[musicxml_path],
    output_file_path=temp_dir / f"{sample['lmxe_path'].stem}.tsv",
  )
  
  return pw_metrics, metrics

pw_metrics, metrics = compare_omr_ned(debug_sample)

  0%|          | 0/4 [00:00<?, ?it/s]

Writing OMR-NED results to CSV:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Writing OMR-NED results to CSV:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
pw_metrics

namespace(output_file_path=PosixPath('/home/dongmin/userdata/dongmin/zeus-omr-pytorch/temp/sq7397765:0007:0005_pw.tsv'),
          macro_average_omr_ned=0.2274949671592212,
          micro_average_omr_ned=0.21696801112656466,
          total_gt_numsyms=719,
          total_pred_numsyms=719,
          total_numsyms=1438,
          total_edit_distance=312,
          total_num_success=4)

In [12]:
metrics

namespace(output_file_path=PosixPath('/home/dongmin/userdata/dongmin/zeus-omr-pytorch/temp/sq7397765:0007:0005.tsv'),
          macro_average_omr_ned=0.21696801112656466,
          micro_average_omr_ned=0.21696801112656466,
          total_gt_numsyms=719,
          total_pred_numsyms=719,
          total_numsyms=1438,
          total_edit_distance=312,
          total_num_success=1)

In [ ]:
diffs = []
for segment in tqdm(test_segments):
  pw_metrics, metrics = compare_omr_ned(segment)
  
  diff_gt_sym = metrics.total_gt_numsyms - pw_metrics.total_gt_numsyms
  diff_pred_sym = metrics.total_pred_numsyms - pw_metrics.total_pred_numsyms
  diffs.append( (diff_gt_sym, diff_pred_sym) )

## Partwise eKern

In [10]:
def parse_kern(kern_str:str, kern_type:str) -> list[str]:
  # filter comment lines
  lines = kern_str.split('\n')
  lines = [
    l
    for l in lines
    if l[:3] != '!!!'
  ]
  
  kern_str = '\n'.join(lines)
  
  # remove numeric values after '='
  kern_str = re.sub("(?<=\=)\d+", "", kern_str)

  # replace special characters with tokens
  kern_str = kern_str.replace(' ', ' <s> ')
  kern_str = kern_str.replace('\t', ' <t> ')
  kern_str = kern_str.replace('\n', ' <b> ')
  # kern_str = kern_str.replace(' /', '')
  # kern_str = kern_str.replace(' \\', '')
  # kern_str = kern_str.replace('·/', '')
  # kern_str = kern_str.replace('·\\', '')

  if kern_type == 'kern':
      kern_str = kern_str.replace('·', '').replace('@', '') # remove all separators
  elif kern_type == 'ekern':
      kern_str = kern_str.replace('·', ' ').replace('@', '') # use only dot separators
  elif kern_type == 'bekern':
      kern_str = kern_str.replace('·', ' ').replace('@', ' ') # use both dot and at separators

  return kern_str.strip().split(" ")

In [11]:
PITCH_TOKENS = [
  'A', 'AA', 
  'a', 'aa', 'aaa'
  
  'B', 'BB', 'BBB', 
  'b', 'bb', 'bbb',
  
  'C', 'CC',
  'c', 'cc', 'ccc', 'cccc',
  
  'D', 'DD',
  'd', 'dd', 'ddd', 'dddd',
  
  'E', 'EE',
  'e', 'ee', 'eee', 'eeee',
  
  'F', 'FF',
  'f', 'ff', 'fff', 'ffff',
  
  'G', 'GG',
  'g', 'gg', 'ggg', 'gggg',
  
]
PITCH_TOKENS = set(PITCH_TOKENS)

STEM_TOKENS = [
  "/",  # up stem
  "\\", # down stem
]
STEM_TOKENS = set(STEM_TOKENS)

ACCIDENTAL_TOKENS = [
  '#', '##', # sharp, double sharp
  '-', '--', # flat, double flat
]
ACCIDENTAL_TOKENS = set(ACCIDENTAL_TOKENS)

SET_TO_TYPE = {
  str(STEM_TOKENS): 'STEM_TOKENS', 
  str(ACCIDENTAL_TOKENS): 'ACCIDENTAL_TOKENS', 
  str(PITCH_TOKENS): 'PITCH_TOKENS', 
}

def get_corrupt_methods(token:str, remove=True) -> str:
  for token_set in [ STEM_TOKENS, ACCIDENTAL_TOKENS, PITCH_TOKENS, ]:
    if token in token_set:
      token_type = SET_TO_TYPE[str(token_set)]
      currupt_methods = list(token_set)
      if remove:
        currupt_methods.append('')
        
      return token_type, currupt_methods
  
  currupt_methods = [token]
  if remove:
    currupt_methods.append('')
  
  return 'UNKNOWN', currupt_methods


def corrupt_sequence(token_seq: list[str], n:int, remove:bool=True, random_seed: Optional[int]=251217) -> list[str]:
  """
  Corrupt n tokens in the token sequence.
  """
  
  if random_seed is not None:
    random.seed(random_seed)
  
  currpted_tokens = [] # list of (index, token_type, original_token, corrupted_token)
  
  token_indices = list(range(len(token_seq)))
  corrupt_indices = random.sample(token_indices, min(n, len(token_indices)))
  
  for idx in corrupt_indices:
    cur = token_seq[idx]
    token_type, corrupt_methods = get_corrupt_methods(cur, remove=remove)
    currupted_token = random.choice(corrupt_methods)
    
    token_seq[idx] = currupted_token
    
    currpted_tokens.append( (idx, token_type, cur, currupted_token) )
  
  return token_seq, currpted_tokens

In [12]:
def compare_omr_ned(sample, *, n=5, remove=False, temp_dir=None):  
  if not temp_dir:
    temp_dir = Path.cwd() / 'temp'
  
  kern_path = temp_dir / sample['kern_path'].with_suffix('.krn').name
  
  # save non-corrupted ekern
  with open(sample['kern_path'], 'r') as f:
    kern_str = f.read()
  
  kern = parse_kern(kern_str, kern_type='kern')
  kern = ''.join(kern)
  kern = kern.replace('<s>', ' ').replace('<t>', '\t').replace('<b>', '\n')
  kern = kern.replace('**ekern', '**kern')
  
  with open(kern_path, 'w') as f:
    f.write(kern)
  
  kern_doc, _ = kp.load(kern_path)
  
  part_ids = kern_doc.get_spine_ids()
  pwkern_paths = []
  for pid, rpid in zip(part_ids, reversed(part_ids)):
    pwkern_path = temp_dir / f"{sample['kern_path'].stem}:{rpid+1}.krn"
    kp.dump(
      kern_doc,
      pwkern_path,
      spine_ids=[pid],
    )
    pwkern_paths.append(pwkern_path)
  
  # save corrupted ekern
  corrupt_kern_path = temp_dir / (sample['kern_path'].stem + '_c.krn')
  
  kern = parse_kern(kern_str, kern_type='bekern')
  kern, _ = corrupt_sequence(kern, n=n, remove=remove)
  kern = ''.join(kern)
  kern = kern.replace('<s>', ' ').replace('<t>', '\t').replace('<b>', '\n')
  kern = kern.replace('**ekern', '**kern')
  with open(corrupt_kern_path, 'w') as f:
    f.write(kern)
  
  kern_doc, _ = kp.load(corrupt_kern_path)
  kp.dump(
    kern_doc,
    corrupt_kern_path,
    encoding=kp.Encoding.normalizedKern,
  )
  
  part_ids = kern_doc.get_spine_ids()
  corrupt_pwkern_paths = [] 
  for pid, rpid in zip(part_ids, reversed(part_ids)):
    corrupt_pwkern_path = temp_dir / f"{sample['kern_path'].stem}:{rpid+1}_c.krn"
    kp.dump(
      kern_doc,
      corrupt_pwkern_path,
      spine_ids=[pid],
    )
    corrupt_pwkern_paths.append(corrupt_pwkern_path)
  
  
  pw_metrics = calc_omr_ned(
    pwkern_paths,
    max_length=0,
    prediction_paths=corrupt_pwkern_paths,
    ground_truth_paths=pwkern_paths,
    output_file_path=temp_dir / f"{sample['lmxe_path'].stem}_pw.tsv",
  )
  
  metrics = calc_omr_ned(
    [kern_path],
    max_length=0,
    prediction_paths=[corrupt_kern_path],
    ground_truth_paths=[kern_path],
    output_file_path=temp_dir / f"{sample['lmxe_path'].stem}.tsv",
  )
  
  return pw_metrics, metrics

pw_metrics, metrics = compare_omr_ned(debug_sample)
# compare_omr_ned(debug_sample)

  0%|          | 0/4 [00:00<?, ?it/s]

Writing OMR-NED results to CSV:   0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

Writing OMR-NED results to CSV:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
pw_metrics

namespace(output_file_path=PosixPath('/home/dongmin/userdata/open-score-string-quartets/temp/sq7397765:0013:0002_pw.tsv'),
          macro_average_omr_ned=0.0,
          micro_average_omr_ned=0.0,
          total_gt_numsyms=417,
          total_pred_numsyms=417,
          total_numsyms=834,
          total_edit_distance=0,
          total_num_success=4)

In [14]:
metrics

namespace(output_file_path=PosixPath('/home/dongmin/userdata/open-score-string-quartets/temp/sq7397765:0013:0002.tsv'),
          macro_average_omr_ned=0.0,
          micro_average_omr_ned=0.0,
          total_gt_numsyms=417,
          total_pred_numsyms=417,
          total_numsyms=834,
          total_edit_distance=0,
          total_num_success=1)

In [ ]:
metrics_pair = []
for segment in tqdm(test_segments):
  pw_metrics, metrics = compare_omr_ned(segment)
  metrics_pair.append( (pw_metrics, metrics) )

diffs = []
for pw_metrics, metrics in metrics_pair:
  diff_gt_sym = metrics.total_gt_numsyms - pw_metrics.total_gt_numsyms
  diff_pred_sym = metrics.total_pred_numsyms - pw_metrics.total_pred_numsyms
  
  if diff_gt_sym != 0 or diff_pred_sym != 0:
    diffs.append( (diff_gt_sym, diff_pred_sym) )

len(diffs)

In [18]:
len(diffs)

0